In [1]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd().resolve()

if project_root.name == "notebooks":
    project_root = project_root.parent

data_file = project_root / "data" / "raw" / "retail_sales_dataset.xlsx"

source_tables = {
    "Customers": pd.read_excel(data_file, sheet_name="Customers"),
    "Products": pd.read_excel(data_file, sheet_name="Products"),
    "Stores": pd.read_excel(data_file, sheet_name="Stores"),
    "Transactions": pd.read_excel(data_file, sheet_name="Transactions")
}

for name, table in source_tables.items():
    print(f"{name}: {table.shape}")
    print(table.head(2))
    print("-" * 80)

Customers: (200, 7)
  CustomerID FirstName LastName Gender  BirthDate               City  \
0       C001   Michael    Davis      M 1996-09-11        Osborneport   
1       C002   Michael   Miller      M 1959-08-18  New Gabrielleport   

    JoinDate  
0 2022-09-25  
1 2020-11-03  
--------------------------------------------------------------------------------
Products: (50, 6)
  ProductID          ProductName     Category SubCategory  UnitPrice  \
0      P001          Like Camera  Electronics      Camera    1673.69   
1      P002  Audience Television  Electronics  Television     818.76   

   CostPrice  
0    1323.38  
1     527.62  
--------------------------------------------------------------------------------
Stores: (5, 4)
  StoreID                StoreName            City Region
0    S001  MegaMart Jimenezborough  Jimenezborough  South
1    S002       MegaMart Peckmouth       Peckmouth   East
--------------------------------------------------------------------------------
Transa

In [2]:
customers = source_tables["Customers"].copy()
products = source_tables["Products"].copy()
stores = source_tables["Stores"].copy()
transactions = source_tables["Transactions"].copy()

customers["BirthDate"] = pd.to_datetime(customers["BirthDate"], errors="coerce")
customers["JoinDate"] = pd.to_datetime(customers["JoinDate"], errors="coerce")
transactions["Date"] = pd.to_datetime(transactions["Date"], errors="coerce")

analytical_df = transactions.merge(customers, on="CustomerID", how="left")
analytical_df = analytical_df.merge(products, on="ProductID", how="left")
analytical_df = analytical_df.merge(stores, on="StoreID", how="left", suffixes=("_Customer", "_Store"))

analytical_df = analytical_df.rename(
    columns={
        "Date": "TransactionDate",
        "City_Customer": "CustomerCity",
        "City_Store": "StoreCity",
        "Region": "StoreRegion"
    }
)

analytical_df["gross_sales"] = analytical_df["Quantity"] * analytical_df["UnitPrice"]
analytical_df["discount_amount"] = analytical_df["gross_sales"] * analytical_df["Discount"]
analytical_df["net_sales"] = analytical_df["gross_sales"] - analytical_df["discount_amount"]
analytical_df["total_cost"] = analytical_df["Quantity"] * analytical_df["CostPrice"]
analytical_df["profit"] = analytical_df["net_sales"] - analytical_df["total_cost"]

analytical_df["year"] = analytical_df["TransactionDate"].dt.year
analytical_df["month"] = analytical_df["TransactionDate"].dt.month
analytical_df["year_month"] = analytical_df["TransactionDate"].dt.to_period("M").astype(str)

analytical_df.head()

,TransactionID,TransactionDate,CustomerID,ProductID,StoreID,Quantity,Discount,PaymentMethod,FirstName,LastName,...,StoreCity,StoreRegion,gross_sales,discount_amount,net_sales,total_cost,profit,year,month,year_month
0,T00001,2024-06-18,C160,P014,S003,1,0.10,Bank Transfer,Meagan,Macdonald,...,New Michele,West,1342.75,134.275,1208.475,797.94,410.535,2024,6,2024-06
1,T00002,2023-11-02,C171,P030,S004,3,0.15,Bank Transfer,Christina,Dominguez,...,Brianahaven,North,87.72,13.158,74.562,45.84,28.722,2023,11,2023-11
2,T00003,2024-03-28,C142,P002,S002,2,0.15,Mobile Money,Suzanne,Fox,...,Peckmouth,East,1637.52,245.628,1391.892,1055.24,336.652,2024,3,2024-03
3,T00004,2024-06-15,C174,P050,S002,5,0.10,Mobile Money,Scott,Howell,...,Peckmouth,East,5223.20,522.320,4700.880,3875.35,825.530,2024,6,2024-06
4,T00005,2024-08-29,C141,P036,S001,3,0.10,Credit Card,Adam,Lucas,...,Jimenezborough,South,4504.38,450.438,4053.942,3503.19,550.752,2024,8,2024-08


In [3]:
print("Dimensión final de la tabla analítica:")
print(analytical_df.shape)

print("\nColumnas disponibles:")
print(analytical_df.columns.tolist())

print("\nResumen rápido:")
print(f"Transacciones: {analytical_df['TransactionID'].nunique()}")
print(f"Ventas netas totales: {analytical_df['net_sales'].sum():,.2f}")
print(f"Utilidad total: {analytical_df['profit'].sum():,.2f}")
print(f"Ticket promedio: {analytical_df['net_sales'].mean():,.2f}")

Dimensión final de la tabla analítica:
(5000, 30)

Columnas disponibles:
['TransactionID', 'TransactionDate', 'CustomerID', 'ProductID', 'StoreID', 'Quantity', 'Discount', 'PaymentMethod', 'FirstName', 'LastName', 'Gender', 'BirthDate', 'CustomerCity', 'JoinDate', 'ProductName', 'Category', 'SubCategory', 'UnitPrice', 'CostPrice', 'StoreName', 'StoreCity', 'StoreRegion', 'gross_sales', 'discount_amount', 'net_sales', 'total_cost', 'profit', 'year', 'month', 'year_month']

Resumen rápido:
Transacciones: 5000
Ventas netas totales: 14,301,903.15
Utilidad total: 3,826,314.67
Ticket promedio: 2,860.38


In [4]:
monthly_sales = (
    analytical_df.groupby("year_month", as_index=False)["net_sales"]
    .sum()
    .sort_values("year_month")
)

sales_by_category = (
    analytical_df.groupby("Category", as_index=False)["net_sales"]
    .sum()
    .sort_values("net_sales", ascending=False)
)

profit_by_region = (
    analytical_df.groupby("StoreRegion", as_index=False)["profit"]
    .sum()
    .sort_values("profit", ascending=False)
)

top_products = (
    analytical_df.groupby("ProductName", as_index=False)["net_sales"]
    .sum()
    .sort_values("net_sales", ascending=False)
    .head(10)
)

monthly_sales.head()

,year_month,net_sales
0,2023-09,380992.278
1,2023-10,561231.162
2,2023-11,584407.279
3,2023-12,653301.195
4,2024-01,584040.146
